# Derivation of One- and Two-Soliton Solutions to the Korteweg-De Vries Equation using the Hirota Method

## 1. Overview

In the present notebook we consider the Korteweg-De Vries equation

$$
\frac{\partial u}{\partial t} + 6 u \frac{\partial u}{\partial x} + \frac{\partial^3 u}{\partial x^3} = 0
\tag{KdVEq}
$$

and find its 1- and 2-soliton solutions by applying the method proposed by Hirota in his paper 2. and following the approach of Griffiths in 1. We also provide some intermediate results related to analytical and symbolic computations that are not included in 1.

The document is structured as a sequence of cells. Each cell includes SymPy code and its analytical explanation and motivation in the respective cell above. Presumably, it is a good way for a reader to follow the mathematical concepts and to naturally connect it to the code that does all the symbolic computations related to the given concept.

In addition, results from each cell are displayed as nice, easy-to-read expressions using LaTeX. The document ends with an animation showing 1- and 2-soliton solution that move across the horizontal line.

In (KdVEq), $u$ is a function that depends on the independent variables $x$ and $t$, i.e. $u = u(x, t)$. Also, $u$ is a subject to the associated boundary condition $u(x, t) = 0$ at $x = \pm \infty$. 

## 2. Introduction and Applications

### 2.1 Introduction

To introduce the reader to the background and history of the solitons, we provide a selected part of the description by Le Coz in his paper 3.

Solutions arising from various nonlinear equations like Korteweg–de Vries, Klein–Gordon or Kadomtsev–Petviashvili equations represent a general class of solutions. These equations enjoy special solutions whose profile remains unchanged under the evolution in time. These special solutions are called solitary waves or solitons.

This kind of phenomena was discovered in 1834 by John Scott Russel. The Scottish engineer was supervising work in a canal near Edinburgh when he observed that the brutal stop of a boat in the canal was creating a wave that does not seem to vanish. Indeed, he was able to follow this wave horseback on a distance of several miles. His life long, he studied this surprising phenomenon but without being able to give it a theoretical justification. In fact, most of the scientists of that time did believe that such a wave, which does not disperse, could not exist. The first theoretical justification of the existence of solitary waves was given by Korteweg and de Vries in 1895. They derived an equation for the motion of water admitting solitary wave solutions. But one had to wait until the 1950’s to see the beginning of an intensive research from mathematicians and physicists about solitary waves.

Heuristically, such solutions appear because of the balance of two contradictory effects: the dispersive effect of the linear part, which tends to flatten the solution as time goes on, and the focusing effect of the nonlinearity, which tends to concentrate the solution.

### 2.2 Applications

The paper of Crighton 4. provides an overview of different applications of the Korteweg-De Vries equation (KdVEq). Each application is substantiated by mathematical background and physical reasoning. An interested reader can follow the exposition in 4. For the purposes of our document, we will list different applications with concise description of the respective highlights.

- **Surface gravity waves**

  The Korteweg-De Vries equation was first developed to describe long waves moving in shallow waters (for example waves in canals or near coastlines). It explains how a single hump of water can travel without changing its shape, forming what was later called a solitary wave.

 - **Internal solitons in the ocean**

   In the ocean, waves can also move inside the water, along layers with different densities not only on the surface. The Korteweg-De Vries equation helps describe these internal waves, which play an important role in ocean mixing and energy transport.


 - **Non-linear acoustics of bubbly liquids**

   When a liquid contains many small gas bubbles, sound waves behave differently from ordinary sound in water. The interaction between pressure and bubble motion allows sound pulses to travel as stable solitary waves described by the Korteweg-De Vries equation.


 - **Voidage slugs in fluidized beds**

   In industrial systems where solid particles are suspended by flowing gas or liquid, regions with fewer particles can move through the material like waves. The Korteweg-De Vries equation models how these concentration waves travel while keeping a stable form.


 - **Magma flow and conduit waves**

   Magma moving through volcanic channels can develop pressure disturbances that travel along the conduit. Under certain conditions, these disturbances behave like solitary waves and can be described using models of the type of Korteweg-De Vries equation.


 - **Jupiter's atmosphere**

   Large-scale wave structures observed in planetary atmospheres, such as those on Jupiter, can sometimes behave like solitons. Simplified atmospheric models show that their motion may follow equations similar to the Korteweg-De Vries equation.


 - **Plasma physics**

   In plasmas (ionized gases), waves caused by interactions between ions and electrons can propagate as stable pulses. The Korteweg-De Vries equation successfully describes these ion–acoustic solitons observed in laboratory and space plasmas.


 - **Electrical Transmission Lines**

   Special electrical circuits containing non-linear components can support voltage pulses that travel without spreading out. These electrical pulses behave like solitons and are mathematically modeled by the Korteweg-De Vries equation.

## 3. Derivation of the 1-Soliton Solution and Implementing the SymPy Code

### 3.1 Set-Up of the Libraries, Variables and Functions that we need to derive the solution

In [1]:
import sympy as sp
from sympy import latex
from IPython.display import Math

In [2]:
# Define all needed variables
x, t, k, c = sp.symbols('x t k c', real=True)

In [3]:
# Define all needed functions
u = sp.Function('u')(x, t)
f = sp.Function('f')(x, t)
g = sp.Function('g')(x, t)

### 3.2 Definition of the Korteweg-De Vries Equation

Here we define the Korteweg-De Vries equation, as set up above in (KdVEq).

In [4]:
# Define the Korteweg-De Vries equation (KdVEq)
kdv_eq = sp.diff(u, t) + 6 * u * sp.diff(u, x) + sp.diff(u, x, 3)
# print(kdv_eq)
Math(latex(kdv_eq))

<IPython.core.display.Math object>

### 3.3 Hirota Derivative

We define the Hirota derivative as given in his paper 2. In parallel, for the purposes of symbolic computations, we implement it as a function that will be used multiple times.

The form of the derivative suggested by Hirota is:
$$
D_x^m D_t^n (f \cdot g) = \left( \frac{\partial}{\partial x} - \frac{\partial}{\partial x'}\right)^m \left( \frac{\partial}{\partial t} - \frac{\partial}{\partial t'}\right)^n f(x, t) \cdot g(x', t') \Bigg\rvert_{x' = x, t' = t}
$$

where $D$ is a binary operator due to the fact that it acts on a pair of functions.

In the case of one variable (say $x$), upon expanding the $m$-th power, $D$ can be written as
$$
D_x^m (f \cdot g) = \sum_{r = 0}^{m} (-1)^r \binom{m}{r} \frac{\partial^{(m - r)} f}{\partial x^{(m  -r)}} \frac{\partial^r g}{\partial x^r},
$$
where $\binom{m}{r}$ is the binomial coefficient defined as $\binom{m}{r} = \frac{m!}{r! (m - r)!}$ for $0 \leq r \leq m$.

Namely, this form of the Hirota derivative is implemented in the function using SymPy.

In [5]:
# Define Hirota derivative in one variable of a given order
def hirota_derivative(func_1, func_2, variable, order):
    return sum(
        ((-1) ** r)
        * sp.binomial(order, r)
        * sp.diff(func_1, variable, order - r)
        * sp.diff(func_2, variable, r)
        for r in range(order + 1)
    )

#### 3.3.1 Hirota Partial Derivative in $t$

Applying the definition of Hirota derivative, it is straight forward to compute analytically $D_t(f \cdot g)$. For the purposes of symbolic computations, we simply employ the defined function above. The results of both analytical and symbolical computations are visible upon running the cell below.

In [6]:
# Define Hirota partial derivative in t
hirota_derivative_t = hirota_derivative(f, g, t, 1)
# print(hirota_derivative_t)
Math(latex(hirota_derivative_t))

<IPython.core.display.Math object>

#### 3.3.2 Mixed Hirota Partial Derivative in $x$ and $t$

To compute analytically the mixed derivative in $x$ and $t$ we apply the definition of $D_x^m D_t^n (f \cdot g)$ above in the case when $n = m = 1$. When performing the differentiation, we take into account the fact that $f$ is a function of $x$ and $t$, while $g$ is a function of $x'$ and $t'$. Eventually, we substitute $x'$ by $x$ and $t'$ by $t$, respectively. 
The symbolic computation is based on the function we defined above. The results of both analytical and symbolical computations are visible upon running the cell below.

In [7]:
# Define Hirota mixed derivative in two variables of first order 
hirota_derivative_x_t = - hirota_derivative(f, sp.diff(g, t, 1), x, 1) + hirota_derivative(sp.diff(f, t, 1), g, x, 1) 
# print(hirota_derivative_x_t)
Math(latex(hirota_derivative_x_t))

<IPython.core.display.Math object>

#### 3.3.3 Hirota Partial Derivative in $x$ of Order $4$

Analytical computation of $D_x^4 (f \cdot g)$ follows the definition when $n = 4$. We expand the summation and compute all binomial coefficients in front of each monomial term. 
Symbolic computation uses the function we defined above. The results of both analytical and symbolical computations are visible upon running the cell below.

In [8]:
# Define Hirota partial derivative in x of order 4
hirota_derivative_x_order_4 = hirota_derivative(f, g, x, 4)
# print(hirota_derivative_x_order_4)
Math(latex(hirota_derivative_x_order_4))

<IPython.core.display.Math object>

#### 3.3.4 Hirota Bilinear Operator

We compute the Hirota bilinear form $(D_x D_t + D_x^4) (f \cdot f)$ in the following way:
- First we use the linearity of $D$ and write $(D_x D_t + D_x^4) (f \cdot g) = (D_x D_t) (f \cdot g) + (D_x^4) (f \cdot g)$.
- Then we take advantage of the results above, where we have already computed $(D_x D_t) (f \cdot g)$ and $(D_x^4) (f \cdot g)$.
- Eventually, we substitute $g$ by $f$.

The symbolic computation is a little easier. We deploy the function we defined above and then simply substitute $g$ by $f$.

The results of both analytical and symbolical computations are visible upon running the cell below.

In [9]:
# Define Hirota bilinear form that is used to solve the KdV equation
hirota_bilinear_form_two_functions = hirota_derivative_x_t + hirota_derivative_x_order_4
hirota_bilinear_form_one_function = hirota_bilinear_form_two_functions.subs(g, f)  # substitue g by f
# print(hirota_bilinear_form_one_function)
Math(latex(hirota_bilinear_form_one_function))

<IPython.core.display.Math object>

#### 3.3.5 Compute an Expression that Involves the Hirota Bilinear Operator

Our goal here is to calculate the following expression

$$
\frac{\partial}{\partial x} \left( \frac{1}{f^2} (D_x D_t + D_x^4) (f \cdot f) \right)
$$

Down below, it will be justified why we needed to calculate this function.

The analytical approach is straigth forward - we already know $(D_x D_t + D_x^4) (f \cdot f)$. So, we just divide it by $f^2$ and compute the partial derivative in $x$ by applying the quotient rule. Calculations are tedious. They do not require any specific reasoning, so we will not disclose them.

Symbolic computations simply use the function we defined above. Then we divide it by $f^2$. And eventually we ask SymPy to compute the partial derivative in $x$, as well as to cancel some terms, so that the expression is as simplified as possible.

The results of both analytical and symbolical computations are visible upon running the cell below.

In [10]:
# Define the expression that contains the Hirota bilinear form divided by f ** 2
kdv_eq_hirota_bilinear_form = sp.diff((hirota_bilinear_form_one_function / (f ** 2)), x)
kdv_eq_hirota_bilinear_form = sp.cancel(kdv_eq_hirota_bilinear_form)
# print(kdv_eq_hirota_bilinear_form)
Math(latex(kdv_eq_hirota_bilinear_form))

<IPython.core.display.Math object>

### 3.4 One-Soliton Solution and the KdV Equation

#### 3.4.1 Assumption of the Form of a Solution

We assume that the solution $u$ of the (KdVEq) has the form

$$
v(x, t) = 2 \frac{\partial^2}{\partial x^2} \ln f(x, t) = \frac{2}{f^2} \left( f \frac{\partial^2 f}{\partial x^2} - \left(\frac{\partial f}{\partial x}\right)^2 \right)
$$

Below is the definition of $v$, which we use to substitute $u$ in the forthcoming symbolic computations.

In [11]:
# Introduce the auxiliary function f, such that v defined below is a solution to the KdV equation 
v = 2 * sp.diff(sp.log(f), x, 2)
# print(v)
Math(latex(v))

<IPython.core.display.Math object>

#### 3.4.2 Asymptotic of Assumed Solution

Using the transformation above and the requirement that $v(x, t) \rightarrow 0$ when $x \rightarrow \pm \infty$ implies $\frac{\partial^2}{\partial x^2} \ln f(x, t) \rightarrow 0$ as $x \rightarrow \pm \infty$. Integrating both sides with respect to $x$ we conclude that $\frac{\partial}{\partial x} \ln f(x, t) = A(t)$, where $A(t)$ is a constant that possibly depends on $t$, but not on $x$. Now, we again integrate both sides with respect to $x$ and obtaint $\ln f(x, t) = A(t) x + B(t)$ or equivalently $f(x, t) = \exp(A(t) x + B(t))$. Just like $A(t)$, $B(t)$ is also a constant that perhaps depends on $t$, but not on $x$.

We need to note that the function $f(x, t)$ is not uniquely determined. Indeed, let $\tilde{f}(x, t) = \exp(A(t) x + B(t)) f(x, t)$. Then $\ln \tilde{f}(x, t) = A(t) x + B(t) + \ln f(x, t)$ and therefore $\frac{\partial^2}{\partial x^2} \ln \tilde{f}(x, t) = \frac{\partial^2}{\partial x^2} \ln f(x, t)$. In this case $v(x, t)$ remains unchanged, i.e. $v(x, t) = \frac{\partial^2}{\partial x^2} \ln \tilde{f}(x, t) = \frac{\partial^2}{\partial x^2} \ln f(x, t)$.

The transformation $f \mapsto \exp(A(t) x + B(t)) f$ leaves the solution invariant. And the function $f$ is defined up to multiplication by exponential factors. This freedom is referred to as gauge invariance. In other words, this allows us to multiply $f$ by a proper factor and introduce an equivalently normalized function satisfying $f(x, t) \rightarrow 1$ when $x \rightarrow \pm \infty$.

In the cell below we provide the code that computes symbolically the solution $f(x, t)$ to $v(x, t) = 2 \frac{\partial^2}{\partial x^2} \ln f(x, t) = 0$. The result from this cell is needed just to demonstrate the derivation of the explicit form of $f(x, t)$ and this result is not required for our computations thereafter. In order for us to not impact the following cells with code, we use $h(x, t)$ instead of $f(x, t)$ and we also set $w(x) = \ln \left( h(x, t) \right)$. By doing so, our goal of finding the form of $h(x, t)$ is equivalent to solving the second-order ordinary differential equation $\frac{\mathrm{d}^2}{\mathrm{d} x^2} w(x) = 0$ and then exponentiating with the derived solution.

In [12]:
# We use h instead of f, i.e. v(x, t) = 2 * (ln(h(x, t)))_xx
# Define the function w(x), such that w(x) = ln(h(x, t)) and treat w(x) as a function of x only
# We solve v(x, t) = 2 * (w(x))_xx = 0, i.e., we solve the second-order ordinary differential equation w_xx = 0
w = sp.Function('w')(x)

# Solve 2 * w(x)_xx = 0 to obtain the solution  w(x) = C1 + C2 * x
w_equation = sp.dsolve(sp.Eq(2 * sp.diff(w, x, 2), 0))

# Extract the right hand side (C1 + C2 * x) from w(x) = C1 + C2 * x
w_rhs = w_equation.rhs

# Rename C1 to B(t) and C2 to A(t), so that they are functions of t and are compliant with the notations from the analytic solution above
A = sp.Function('A')(t)
B = sp.Function('B')(t)

w_renamed_variables = w_rhs.subs(
    {
        sp.Symbol('C1'): B,
        sp.Symbol('C2'): A
    }
)

# Define the function h(x, t)
h = sp.Function('h')(x, t)

# Set up the equation w(x) = ln(h(x, t))
h_w_equation = sp.Eq(sp.log(h), w_renamed_variables)

# Derive the solution h(x, t) from the equation ln(h(x, t)) = A(t) * x + B(t)
h_solution = sp.solve(h_w_equation, h)
Math(latex(*h_solution))

<IPython.core.display.Math object>

#### 3.4.3 Substitution into the KdV Equation

When we substitute the function $v(x, t)$ into the Korteweg-De Vries equation (KdVEq), we obtain the expression that results from running the cell below. The process of derivation is long and technical. For the sake of brevity, here we only provide some of the intermediate results (not explicitly included in the paper by Griffiths) that are needed for the substitution into the (KdVEq):

$$
\frac{\partial v}{\partial t} = \frac{2}{f^3} \left( f^2 \frac{\partial^3 f}{\partial x^2 \partial t} - 2 f \frac{\partial f}{\partial x} \frac{\partial^2 f}{\partial x \partial t} + 2 \frac{\partial f}{\partial t} \left( \frac{\partial f}{\partial x} \right)^2 - f \frac{\partial f}{\partial t} \frac{\partial^2 f}{\partial x^2} \right),
$$

$$
\frac{\partial v}{\partial x} = \frac{2}{f^3} \left( f^2 \frac{\partial^3 f}{\partial x^3} - 3 f \frac{\partial f}{\partial x} \frac{\partial^2 f}{\partial x^2} + 2 \left( \frac{\partial f}{\partial x} \right)^3 \right),
$$

$$
\frac{\partial^2 v}{\partial x^2} = \frac{2}{f^4} \left( f^3 \frac{\partial^4 f}{\partial x^4} +12 f \left( \frac{\partial f}{\partial x} \right)^2 \frac{\partial^2 f}{\partial x^2} - 3 f^2 \left( \frac{\partial^2 f}{\partial x^2} \right)^2 - 4 f^2 \frac{\partial f}{\partial x} \frac{\partial^3 f}{\partial x^3} - 6 \frac{\partial^4 f}{\partial x^4} \right)
$$

$$
\frac{\partial^3 v}{\partial x^3} = \frac{2}{f^5} \left( -5 f^3 \frac{\partial f}{\partial x} \frac{\partial^4 f}{\partial x^4} - 60 f \left(  \frac{\partial f}{\partial x} \right)^3 \frac{\partial^2 f}{\partial x^2} +30 f^2 \frac{\partial f}{\partial x} \left( \frac{\partial^2 f}{\partial x^2} \right)^2 + 20 f^2 \left( \frac{\partial f}{\partial x} \right)^2 \frac{\partial^3 f}{\partial x^3} - 10 f^3 \frac{\partial^2 f}{\partial x^2} \frac{\partial^3 f}{\partial x^3} + f^4 \frac{\partial^5 f}{\partial x^5} + 24 \left( \frac{\partial f}{\partial x} \right)^5 \right).
$$

The symbolic substitution follows the same sequence of steps and produces the left side of the (KdVEq) after inserting $v$ instead of $u$. The results of both analytical and symbolical computations are visible upon running the cell below.

In [13]:
# Substitute the function v (depending on the auxiliary function f) into the KdV equation
kdv_eq_aux_function = kdv_eq.subs(u, v)
kdv_eq_aux_function = kdv_eq_aux_function.doit()  # calculates derivatives
kdv_eq_aux_function = sp.cancel(kdv_eq_aux_function)
# print(kdv_eq_aux_function)
Math(latex(kdv_eq_aux_function))

<IPython.core.display.Math object>

#### 3.4.4 Equivalence Between the KdV Equation and the Expression Having the Hirota Form

We already computed what the Korteweg-De Vries Equation looks like after substituting $u$ by $v$. We also expanded the expression

$$
\frac{\partial}{\partial x} \left( \frac{1}{f^2} (D_x D_t + D_x^4) (f \cdot f) \right).
$$

To assure that our computations so far are correct we use SymPy to verify symbolically that the two results are the same. More precisely, we set and compute the difference. As can be seen from the cell below, this difference is $0$, confirming that our calculations are correct.  

In [14]:
# Verify that the partial derivative in x of expression with nominator Hirota bilinear form and denominator f ** 2
# is equivalent to the KdV equation after substituting u by v
diff = kdv_eq_hirota_bilinear_form - kdv_eq_aux_function
diff = sp.cancel(diff)
# print(diff)
Math(latex(diff))

<IPython.core.display.Math object>

### 3.5 Computing the One-Soliton Solution

#### 3.5.1 Hirota Form and the KdV Equation

We already verified that the Korteweg-De Vries equation is equivalent to

$$
\frac{\partial}{\partial x} \left( \frac{1}{f^2} (D_x D_t + D_x^4) (f \cdot f) \right) = 0.
$$

If we integrate both sides with respect to $x$, we derive $\frac{1}{f^2} (D_x D_t + D_x^4) (f \cdot f) = C(t)$, where $C(t)$ is a function that possibly depends on $t$, but not on $x$. Let us now use the gauge invariance of $f(x, t)$ that we already determined above and normalize $f(x, t)$, such that $f \rightarrow 1$ when $x \rightarrow \pm \infty$. Then all partial derivatives in $(D_x D_t + D_x^4) (f \cdot f)$ will be zero (since any derivative of $f$, when $f \rightarrow 1$, is equal to zero). We also have that $f^2 \rightarrow 1$ and eventually we conclude that $C(t) = 0$. With all this in mind, the equation above is equivalent to 

$$
(D_x D_t + D_x^4) (f \cdot f) = 0.
$$

Next, we compute $(D_x D_t + D_x^4) (1 \cdot f_1 + f_1 \cdot 1)$. The analytical calculation uses the fact that we know $(D_x D_t + D_x^4) (f \cdot g)$ as well as its linearity. First, it is easy to compute that $D_x D_t (f_1 \cdot 1) = D_x D_t (1 \cdot f_1) = \frac{\partial^2 f_1}{\partial x \partial t}$. Using the linearity, we have $D_x D_t (1 \cdot f_1 + f_1 \cdot 1) = D_x D_t (1 \cdot f_1 ) + D_x D_t (f_1 \cdot 1) = 2 \frac{\partial^2 f_1}{\partial x \partial t}$.

In a similar way, $D^4_x (1 \cdot f_1) = D^4_x (f_1 \cdot 1) = \frac{\partial^4 f_1}{\partial x^4}$ and thus $D^4_x (1 \cdot f_1 + f_1 \cdot 1) = D^4_x (1 \cdot f_1) + D^4_x (f_1 \cdot 1) = 2 \frac{\partial^4 f_1}{\partial x^4}$.

Eventually, we have that $(D_x D_t + D_x^4) (1 \cdot f_1 + f_1 \cdot 1) = 2 \left( \frac{\partial^2 f_1}{\partial x \partial t} + \frac{\partial^4 f_1}{\partial x^4} \right)$.

This result is confirmed by the symbolic computation from the cell below. There it uses our function defined above as well as the substitution of $f$ and $g$ by $1$ and $f_1$.

In [15]:
# Apply Hirota form to later find the dispersion relation
f_1 = sp.Function('f_1')(x, t)
bilinear_form_1_soliton = hirota_bilinear_form_two_functions.subs({f: 1, g: f_1}) + hirota_bilinear_form_two_functions.subs({g: 1, f: f_1})
bilinear_form_1_soliton = bilinear_form_1_soliton.doit()
# print(bilinear_form_1_soliton)
Math(latex(bilinear_form_1_soliton))

<IPython.core.display.Math object>

#### 3.5.2 Ansatz and Dispersion Relation

Let us set the ansatz $f_1 = e^{\xi}$, where $\xi = k (x - c t)$.

Now, if we substitute $f = 1 + \epsilon f_1$ into 

$$
v(x, t) = \frac{2}{f^2} \left( f \frac{\partial^2 f}{\partial x^2} - \left(\frac{\partial f}{\partial x}\right)^2 \right),
$$

we obtain

$$
u = \frac{k^2}{2} \operatorname{sech}^2 \left( \frac{k}{2} (x - c t) \right),
$$

where we set $\epsilon = 1$.

Following the exposure of Hirota, $f$ that we defined above, is the 1-soliton solution of the Korteweg-De Vries equation.

Above we obtained that $(D_x D_t + D_x^4) (1 \cdot f_1 + f_1 \cdot 1) = 2 \left( \frac{\partial^2 f_1}{\partial x \partial t} + \frac{\partial^4 f_1}{\partial x^4} \right) = 0$. Now, if we plug $f_1$ into this equation, we conclude that:

$$
k^2 e^{k (x - c t)} (k^2 - c) = 0.
$$

From here, the so called dispersion relation is extracted. Namely, $c = k^2$.

The code from the cell below, computes symbolically this relation by substituting the 1-soliton solution into the Hirota form.

In [16]:
# Determine the dispersion relation
f_1_exp = sp.exp(k * (x - c * t))
bilinear_form_1_soliton_exponent = bilinear_form_1_soliton.subs(f_1, f_1_exp)
bilinear_form_1_soliton_exponent = bilinear_form_1_soliton_exponent.doit()
bilinear_form_1_soliton_exponent = sp.simplify(bilinear_form_1_soliton_exponent)
dispersion_relation = sp.solve(bilinear_form_1_soliton_exponent, c)
# print(dispersion_relation)
Math(latex(dispersion_relation))

<IPython.core.display.Math object>

#### 3.5.3 First Result of the 1-Soliton Solution

As explained above, we computed the 1-soliton solution $u = \frac{k^2}{2} \operatorname{sech}^2 \left( \frac{k}{2} (x - c t) \right)$. Here we provide the analytical result and also compute it symbolically in the cell below.

In [17]:
# Define the form of the 1-soliton solution and derive the its explicit according to the Hirota approach
F = 1 + f_1_exp
u_soliton_solution = v.subs(f, F)
u_soliton_solution = u_soliton_solution.doit()
u_soliton_solution = sp.simplify(u_soliton_solution)
u_solution_sech = u_soliton_solution.rewrite(sp.sech)  # rewrite the soliton solution using sech 
# print(u_solution_sech)
Math(latex(u_solution_sech))

<IPython.core.display.Math object>

#### 3.5.4 Final Outcome of the 1-Soliton Solution

After deriving the 1-soliton solution and the dispersion relation, here we only plug $k^2$ instead of $c$ to conclude that

$$
u = \frac{k^2}{2} \operatorname{sech}^2 \left( \frac{k}{2} (x - k^2 t) \right)
$$

In [18]:
# Substitute the dispersion relation into the 1-soliton equation
u_solution = u_solution_sech.subs(c, dispersion_relation[0])
# print(u_solution)
Math(latex(u_solution))

<IPython.core.display.Math object>

### 3.6 Verification of the One-Soliton Solution

We calculated the explicit form of the 1-soliton solution. The goal here is to formally confirm that this expression in fact fulfills the Korteweg-De Vries equation. What we do is simply take $u = \frac{k^2}{2} \operatorname{sech}^2 \left( \frac{k}{2} (x - k^2 t) \right)$ and plug it in the left hand side of (KdVEq) to verify that the result is $0$.

Therefore, here we only apply symbolic computation and confirm that the solution we found actually provides equivalence in the Korteweg-De Vries.

In [19]:
# Verify that the obtained 1-soliton solution solves the KdV equation
verify_kdv_eq = kdv_eq.subs(u, u_solution)
verify_kdv_eq = verify_kdv_eq.doit()
verify_kdv_eq = sp.simplify(verify_kdv_eq)
# print(verify_kdv_eq)
Math(latex(verify_kdv_eq))

<IPython.core.display.Math object>

### 3.7 Asymptotics of the One-Soliton Solution

After finding the 1-soliton solution, we verified that indeed it fulfills the Korteweg-De Vries equation. Now, we would like to check if this solution also obeys the asymptotic assumption $u(x, t) \rightarrow 0$ whenever $x \rightarrow \pm \infty$, as stipulated at the top of this notebook.

As already determined, our solution is

$$
u(x, t) = \frac{k^2}{2} \operatorname{sech}^2 \left( \frac{k}{2} (x - k^2 t) \right),
$$

but for the purpose of computing the limits we will re-write it in the equivalent form

$$
u(x, t) = \frac{2 k^2}{\left( \exp\left( \frac{k}{2} (x - k^2 t) \right) + \exp\left( -\frac{k}{2} (x - k^2 t) \right) \right)^2},
$$

where we used the definition of $\operatorname{sech}(y) = \frac{2}{\exp(y) + \exp(-y)}$ and the exponential function $\exp(\cdot)$.

Let us first note that when $k = 0$, then $u(x, t) = 0$ and we do not need to compute any asymptotics (either at infinity or minus infinity).

Let us now compute the asymptotic of $u$ when $x \rightarrow +\infty$. We will do it by taking into account each of the two cases below.

1) $k > 0$

   Since $x \rightarrow +\infty$ and $k$ is positive, then $\frac{k}{2} > 0$ and $-\frac{k}{2} < 0$. Hence $\exp\left( \frac{k}{2} (x - k^2 t) \right) \rightarrow +\infty$ and $\exp\left( -\frac{k}{2} (x - k^2 t) \right) \rightarrow 0$. And then $u \rightarrow 0$.
2) $k < 0$

   Since $x \rightarrow +\infty$ and $k$ is negative, then $\frac{k}{2} < 0$ and $-\frac{k}{2} > 0$. Hence $\exp\left( \frac{k}{2} (x - k^2 t) \right) \rightarrow 0$ and $\exp\left( -\frac{k}{2} (x - k^2 t) \right) \rightarrow +\infty$. And again $u \rightarrow 0$, as desired.

The case $x \rightarrow -\infty$ is considered in a completely analagous way by taking into account the two options $k > 0$ and $k < 0$. By doing so, we also verify that $u \rightarrow 0$.

In the cells below we perform the symbolic computations of finding the limits by following the analytical solution outlined here.

In [20]:
# When k = 0 no matter if x goes to plus or minus infinity
u_k_zero = u_solution.subs(k, 0)
# print(u_k_zero)
display(Math(latex(u_k_zero)))

<IPython.core.display.Math object>

In [21]:
# x tends to plus infinity
# 1) When k > 0
k_1 = sp.symbols('k_1', positive=True)
u_x_plus_infinity_k_positive = u_solution.subs(k, k_1)
plus_infinity_limit_k_positive = sp.limit(u_x_plus_infinity_k_positive, x, sp.oo)
# print(plus_infinity_limit_k_positive)
display(Math(latex(plus_infinity_limit_k_positive)))

# 2) When k < 0
k_2 = sp.symbols('k_2', negative=True)
u_x_plus_infinity_k_negative = u_solution.subs(k, k_2)
plus_infinity_limit_k_negative = sp.limit(u_x_plus_infinity_k_negative, x, sp.oo)
# print(plus_infinity_limit_k_negative)
display(Math(latex(plus_infinity_limit_k_negative)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [22]:
# x tends to miinus infinity
# 3) When k > 0
k_3 = sp.symbols('k_3', positive=True)
u_x_minus_infinity_k_positive = u_solution.subs(k, k_3)
minus_infinity_limit_k_positive = sp.limit(u_x_minus_infinity_k_positive, x, -sp.oo)
# print(minus_infinity_limit_k_positive)
display(Math(latex(minus_infinity_limit_k_positive)))

# 4) When k < 0
k_4 = sp.symbols('k_4', negative=True)
u_x_minus_infinity_k_negative = u_solution.subs(k, k_4)
minus_infinity_limit_k_negative = sp.limit(u_x_minus_infinity_k_negative, x, -sp.oo)
# print(minus_infinity_limit_k_negative)
display(Math(latex(minus_infinity_limit_k_negative)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 3.8 Computing the Two-Soliton Solution

#### 3.8.1 Ansatz and Dispersion Relations

Derivation of the 2-soliton solution follows the same process as the derivation of the 1-soliton solution. In particular, for the 2-soliton solution we set the ansatz $f_1 = e^{\xi_1} + e^{\xi_2}$, $f_2 = a_{12} e^{\xi_1 + \xi_2}$, where $a_{12}$ is a constant that we will determine and $\xi_1 = k_1 (x - c_1 t)$, $\xi_2 = k_2 (x - c_2 t)$. We also define the auxiliary function $f = 1 + \epsilon f_1 + \epsilon^2 f_2$.

Above we derived the following relation about the bilinear form $(D_x D_t + D_x^4) (1 \cdot f_1 + f_1 \cdot 1) = 2 \left( \frac{\partial^2 f_1}{\partial x \partial t} + \frac{\partial^4 f_1}{\partial x^4} \right)$. Now setting $\epsilon = 1$ we consecutively compute that $\frac{\partial^2 f_1}{\partial x \partial t} = -k_1^2 c_1 e^{\xi_1} - k_2^2 c_2 e^{\xi_2}$ and $\frac{\partial^4 f_1}{\partial x^4} = k_1^4 e^{\xi_1} + k_2^4 e^{\xi_2}$. Further, we know that $(D_x D_t + D_x^4) (1 \cdot f_1 + f_1 \cdot 1) = 0$ and substituting the partial drivatives of $f_1$ we determine the relation $2 \left( k_1^2 (k_1^2 - c_1) e^{\xi_1} + k_2^2 (k_2^2 - c_2) e^{\xi_2} \right) = 0$. There are several ways to conclude that $c_1 = k_1^2$ and $c_2 = k_2^2$. We list them below.

1) The functions $e^{\xi_1}$ and $e^{\xi_2}$ are linearly independent. They correspond to different and independent soliton waves. The relation that we obtained is a linear combination (of these two functions) equal to zero. The only option to have it fulfilled is to have each coefficient equal to zero.
2) As mentioned above, $e^{\xi_1}$ and $e^{\xi_2}$ are linearly independent and we can plug each of them separately in the relation of bilinear form equal to zero. This will result in two separate equations: $2 k_1^2 (k_1^2 - c_1) e^{\xi_1} = 0$ and $2 k_2^2 (k_2^2 - c_2) e^{\xi_2} = 0$.
3) Without taking into account the fact that $e^{\xi_1}$ and $e^{\xi_2}$ are linearly independent, we can consider the case when both $k_1$ and $k_2$ are positive and $k_1^2 > c_1$ and $k_2^2 > c_2$. Then the left side $k_1^2 (k_1^2 - c_1) e^{\xi_1} + k_2^2 (k_2^2 - c_2) e^{\xi_2} = 0$ will go to $+\infty$ and will apparently be different from $0$. Moreover, the relation needs to be fulfilled always. This can happen only when the exponential terms are eliminated, meaning that we derive the two dispersion relations above.

The code from the cell below, computes symbolically the two dispersion relations by substituting the 1-soliton solution into the Hirota form. The result is in the form of a dictionary, where the keys are $c_1$ and $c_2$, and the values are $k_1^2$ and $k_2^2$.

In [23]:
# Determine the two dispersion relations
k1, k2, c1, c2 = sp.symbols('k1 k2 c1 c2', real=True)
xi1 = k1 * (x - c1 * t) 
xi2 = k2 * (x - c2 * t)
exponent1 = sp.exp(xi1)
exponent2 = sp.exp(xi2)

f1 = exponent1 + exponent2

bf_1_soliton_exponent = bilinear_form_1_soliton.subs(f_1, f1)
bf_1_soliton_exponent = bf_1_soliton_exponent.doit()

exponents = list(bf_1_soliton_exponent.atoms(sp.exp))
bf_1_soliton_exponent = sp.collect(bf_1_soliton_exponent, [exponent1, exponent2])
coefficients = [bf_1_soliton_exponent.coeff(e) for e in exponents]
dispersion_relations = sp.solve(coefficients, [c1, c2])

# print(dispersion_relations)
Math(latex(dispersion_relations))

<IPython.core.display.Math object>

#### 3.8.2 Relations Stemming from the Bilinear Form

Before moving forward let us elaborate on the relations that are derived from the bilinear form. We limit our considerations to the case of the 2-soliton solution. If we take a general bilinear form $B$, not only the one we consider $(D_x D_t + D_x^4) (f \cdot f)$, we can consider $B(f \cdot f) = 0$. More precisely, on the left hand side, we can group the results by each power of $\epsilon$. By doing so, we will obtain the following: for $\epsilon^0$ we have $B(1 \cdot 1) = 0$, for $\epsilon^1$ we have $B(1 \cdot f_1 + f_1 \cdot 1) = 0$ and for $\epsilon^2$ we have $B(1 \cdot f_2 + f_1 \cdot f_1 + f_2 \cdot 1) = 0$.

We have already employed the $B(1 \cdot f_1 + f_1 \cdot 1) = 0$ in the cases of 1- and 2-soliton solutions. Now we provide more insights how and where we derive it from.

#### 3.8.3 Calculate the Constant $a_{12}$ from the Bilinear Relations

To calculate the constant $a_{12}$ we take advantage of the relation $(D_x D_t + D_x^4) (1 \cdot f_2 + f_1 \cdot f_1 + f_2 \cdot 1) = 0$. Due to the linearity of the form we can write $(D_x D_t + D_x^4) (1 \cdot f_2 + f_2 \cdot 1) + (D_x D_t + D_x^4) (f_1 \cdot f_1) = 0$. We already know that $(D_x D_t + D_x^4) (1 \cdot f_2 + f_2 \cdot 1) = 2 \left( \frac{\partial^2 f_2}{\partial x \partial t} + \frac{\partial^4 f_2}{\partial x^4} \right)$ and we need to compute $(D_x D_t + D_x^4) (f_1 \cdot f_1)$. Again, taking into account the linearity of the bilinear form, we can write $(D_x D_t + D_x^4) (f_1 \cdot f_1) = D_x D_t (f_1 \cdot f_1) + D_x^4 (f_1 \cdot f_1)$.

Applying the Hirota derivative consecutively first in respect to $t$ and then to $x$ we obtain $D_x D_t (f_1 \cdot f_1) = 2\left( f_1 \frac{\partial^2 f_1}{\partial x \partial t} - \frac{\partial f_1}{\partial x} \frac{\partial f_1}{\partial t} \right)$. In an analogous way we apply the definition of the Hirota derivative four times and thus compute $D_x^4 (f_1 \cdot f_1) = 2 f_1 \frac{\partial^4 f_1}{\partial x^4} - 8 \frac{\partial f_1}{\partial x} \frac{\partial^3 f_1}{\partial x^3} + 6 \left( \frac{\partial^2 f_1}{\partial x^2} \right)^2$. Therefore, we have 

$$
(D_x D_t + D_x^4) (f_1 \cdot f_1) = D_x D_t (f_1 \cdot f_1) + D_x^4 (f_1 \cdot f_1) = 2\left( f_1 \frac{\partial^2 f_1}{\partial x \partial t} - \frac{\partial f_1}{\partial x} \frac{\partial f_1}{\partial t} \right) + 2 f_1 \frac{\partial^4 f_1}{\partial x^4} - 8 \frac{\partial f_1}{\partial x} \frac{\partial^3 f_1}{\partial x^3} + 6 \left( \frac{\partial^2 f_1}{\partial x^2} \right)^2
$$

$$
= 2 f_1 \left( \frac{\partial^2 f_1}{\partial x \partial t} + \frac{\partial^4 f_1}{\partial x^4} \right) - 2 \frac{\partial f_1}{\partial x} \frac{\partial f_1}{\partial t} - 8 \frac{\partial f_1}{\partial x} \frac{\partial^3 f_1}{\partial x^3} + 6 \left( \frac{\partial^2 f_1}{\partial x^2} \right)^2
$$

$$
= f_1 (D_x D_t + D_x^4) (1 \cdot f_1 + f_1 \cdot 1) - 2 \frac{\partial f_1}{\partial x} \frac{\partial f_1}{\partial t} - 8 \frac{\partial f_1}{\partial x} \frac{\partial^3 f_1}{\partial x^3} + 6 \left( \frac{\partial^2 f_1}{\partial x^2} \right)^2
$$

$$
= - 2 \frac{\partial f_1}{\partial x} \frac{\partial f_1}{\partial t} - 8 \frac{\partial f_1}{\partial x} \frac{\partial^3 f_1}{\partial x^3} + 6 \left( \frac{\partial^2 f_1}{\partial x^2} \right)^2,
$$

where we have used the fact that $(D_x D_t + D_x^4) (1 \cdot f_1 + f_1 \cdot 1) = 2 \left( \frac{\partial^2 f_1}{\partial x \partial t} + \frac{\partial^4 f_1}{\partial x^4} \right) = 0$.

Hence, we compute 

$$
(D_x D_t + D_x^4) (1 \cdot f_2 + f_1 \cdot f_1 + f_2 \cdot 1) = 2 \left( \frac{\partial^2 f_2}{\partial x \partial t} + \frac{\partial^4 f_2}{\partial x^4} \right) - 2 \frac{\partial f_1}{\partial x} \frac{\partial f_1}{\partial t} - 8 \frac{\partial f_1}{\partial x} \frac{\partial^3 f_1}{\partial x^3} + 6 \left( \frac{\partial^2 f_1}{\partial x^2} \right)^2.
$$

Then equating $(D_x D_t + D_x^4) (1 \cdot f_2 + f_1 \cdot f_1 + f_2 \cdot 1) = 0$, we obtain

$$
\frac{\partial^2 f_2}{\partial x \partial t} + \frac{\partial^4 f_2}{\partial x^4} - \frac{\partial f_1}{\partial x} \frac{\partial f_1}{\partial t} - 4 \frac{\partial f_1}{\partial x} \frac{\partial^3 f_1}{\partial x^3} + 3 \left( \frac{\partial^2 f_1}{\partial x^2} \right)^2 = 0.
$$

Next, we compute all derivatives

$$
\frac{\partial^2 f_2}{\partial x \partial t} = -a_{12} (k_1 + k_2) (k_1 c_1 + k_2 c_2) e^{\xi_1 + \xi_2}, \ \ \frac{\partial^4 f_2}{\partial x^4} = a_{12} (k_1 + k_4)^4 e^{\xi_1 + \xi_2}, \ \ \frac{\partial f_1}{\partial t} = -k_1 c_1 e^{\xi_1} - k_2 c_2 e^{\xi_2}
$$

$$
\frac{\partial f_1}{\partial x} = k_1 e^{\xi_1} + k_2 e^{\xi_2}, \ \ \frac{\partial^2 f_1}{\partial x^2} = k_1^2 e^{\xi_1} + k_2^2 e^{\xi_2}, \ \ \frac{\partial^3 f_1}{\partial x^3} = k_1^3 e^{\xi_1} + k_2^3 e^{\xi_2}.
$$

We also keep in mind the dispersion relations $c_1 = k_1^2$ and $c_2 = k_2^2$.

After substituting all partial variables in the relation above and simplifying the expression, we compute that $a_{12} = \frac{(k_1 - k_2)^2}{(k_1 + k_2)^2}$, when $k_1 \neq -k_2$, $k_1 \neq 0$ and $k_2 \neq 0$.

The code from the cell below, computes symbolically the constant $a_{12}$ by applying the steps we outline above. 

In [24]:
# Calculate the constant a12
a12 = sp.symbols('a12', real=True)
f2 = a12 * sp.exp(xi1 + xi2)

bilinear_form_2_soliton = (
    hirota_bilinear_form_two_functions.subs({f: 1, g: f2}) 
    + hirota_bilinear_form_two_functions.subs({g: 1, f: f2}) 
    + hirota_bilinear_form_two_functions.subs({g: f1, f: f1})
)

bilinear_form_2_soliton = bilinear_form_2_soliton.doit()
bilinear_form_2_soliton = bilinear_form_2_soliton.subs({c1: k1 ** 2, c2: k2 ** 2})
bilinear_form_2_soliton = sp.simplify(bilinear_form_2_soliton)

a12_result = sp.solve(bilinear_form_2_soliton, a12)
a12_result = sp.factor(*a12_result)

# print(constant_a12)
Math(latex(a12_result))

<IPython.core.display.Math object>

#### 3.8.4 Calculate the 2-Soliton Solution

Now, if we substitute $f = 1 + \epsilon f_1 + \epsilon^2 f_2$ into 

$$
u(x, t) = \frac{2}{f^2} \left( f \frac{\partial^2 f}{\partial x^2} - \left(\frac{\partial f}{\partial x}\right)^2 \right),
$$

we obtain

$$
u() = 2 \frac{k_1^2 e^{\xi_1} + k_2^2 e^{\xi_2} + a_{12} (k_1 + k_2)^2 e^{\xi_1 + \xi_2}}{1 + e^{\xi_1} + e^{\xi_2} + a_{12} e^{\xi_1 + \xi_2}} - 2 \frac{\left( k_1 e^{\xi_1} + k_2 e^{\xi_2} + a_{12} (k_1 + k_2) e^{\xi_1 + \xi_2} \right)^2}{\left( 1 + e^{\xi_1} + e^{\xi_2} + a_{12} e^{\xi_1 + \xi_2} \right)^2},
$$

where we set $\epsilon = 1$.

The cell below computes symbolically the 2-soliton solution. However, we substitute $\xi_1$, $\xi_2$ and $a_{12}$ with their corresponding values. Hence the result produced by the code is visually different from the form of $u$ above and also contains $x$ and $t$. However, further down, we verify that the solution generated by the code is in fact a solution of (KdVEq).

In [25]:
epsilon = sp.symbols('epsilon', real=True)

f_2_sol = 1 + epsilon * f1 + epsilon ** 2 * f2

f_2_sol = f_2_sol.subs(
    {
        epsilon: 1,
        c1: dispersion_relations[c1], 
        c2: dispersion_relations[c2], 
        a12: a12_result
    }
)

u_2_sol = 2 * sp.diff(sp.log(f_2_sol), x, 2)
u_2_sol = sp.factor(u_2_sol)

# print(u_2_sol)
Math(latex(u_2_sol))

<IPython.core.display.Math object>

#### 3.8.5 Verification of the Two-Soliton Solution

We calculated the explicit form of the 2-soliton solution. The goal here is to formally confirm that this expression fulfills the Korteweg-De Vries equation. What we do is simply take the 2-soliton solution $u(x, t)$ that we calculated and plug it in the left hand side of (KdVEq) to verify that the result is $0$.

Hence, in the cell below we only apply symbolic computation and confirm that the solution we found actually provides equivalence in the Korteweg-De Vries equation.

Please keep in mind that the symbolic code below takes a little more time to run, since the expression is complex and computations are heavy.

In [ ]:
# Verify that the obtained 2-soliton solution solves the KdV equation
# Please be advised that running this cell takes a little longer due to the heavy computations

verify_kdv_eq_2_sol = kdv_eq.subs(u, u_2_sol)
verify_kdv_eq_2_sol = verify_kdv_eq_2_sol.doit()
verify_kdv_eq_2_sol = sp.simplify(verify_kdv_eq_2_sol)

# print(verify_kdv_eq_2_sol)
Math(latex(verify_kdv_eq_2_sol))


## 4. Animation of the 1-Soliton Solution

The code from the cell below generates an animated visualisation of the 1-soliton solution of the Korteweg-De Vries equation (KdVEq). It was obtained by applying the Hirota method and after fixing the parameter $k$ equal to $\sqrt{2}$.

The symbolic solution that we derived above, is converted into a fast numerical function and evaluated in the interval $x \in [-50, 50]$. On the other hand, the time $t$ takes values from the interval $[-20, 20]$. The waves are updated after generating each frame.

The animation shows a soliton travelling in time without changing its shape.

In [ ]:
# Create an animation of travelling 1-soliton solution
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

u_soliton_solution_numeric = u_solution.subs(k, np.sqrt(2))
u_soliton_solution_function = sp.lambdify((x, t), u_soliton_solution_numeric, 'numpy')

figure, axes = plt.subplots()
x_values = np.linspace(-50, 50, 400)
line = axes.plot([], [], lw=3)[0]

axes.set_xlim(-50, 50)
axes.set_ylim(0, 3)
axes.set_xlabel('x')
axes.set_ylabel('u(x, t)')
axes.set_title('KdV 1-Soliton Solution Applying Hirota Method')

def update(time):
    t_value = time
    y = u_soliton_solution_function(x_values, t_value)
    line.set_data(x_values, y)
    return [line]

plt.close(figure)

from IPython.display import HTML

animation = FuncAnimation(
    figure,
    update,
    frames=np.linspace(-20, 20, 120)
)

HTML(animation.to_jshtml())

## 5. Animation of the 2-Soliton Solution

Just like the case of 1-soliton solution, the code from the cell below generates an animated visualisation of the 2-soliton solution of the Korteweg-De Vries equation (KdVEq). It was obtained by applying the Hirota method and after setting the parameter $k_1$ equal to $1$ and the parameter $k_2$ equal to $\sqrt{2}$. The values for $x$ and $t$ are the same as in the 1-soliton plot.

In this case the visualisation shows two solitons travelling in time, interacting and then re-emerging with their original shapes and velocities, but at different positions before the interaction.

In [ ]:
# Create an animation of travelling 2-soliton solution
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

u_2_soliton_solution_numeric = u_2_sol.subs({k1: 1, k2: np.sqrt(2)})
u_2_soliton_solution_function = sp.lambdify((x, t), u_2_soliton_solution_numeric, 'numpy')

figure, axes = plt.subplots()
x_values = np.linspace(-50, 50, 400)
line = axes.plot([], [], lw=3)[0]

axes.set_xlim(-50, 50)
axes.set_ylim(0, 3)
axes.set_xlabel('x')
axes.set_ylabel('u(x, t)')
axes.set_title('KdV 2-Soliton Solution Applying Hirota Method')

def update(time):
    t_value = time
    y = u_2_soliton_solution_function(x_values, t_value)
    line.set_data(x_values, y)
    return [line]

plt.close(figure)

from IPython.display import HTML

animation = FuncAnimation(
    figure,
    update,
    frames=np.linspace(-20, 20, 120)
)

HTML(animation.to_jshtml())

### Bibliography
1) Graham W Griffiths, Hirota Direct Method, City University, UK (https://pdecomp.net/generalPapers/HirotaMethod.pdf)
2) Hirota, R. (1971), Exact solution of the Korteweg-de Vries equation for multiple collisions of solitons, Phys. Review Letters, 27, 1192-1194
3) Stefan Le Coz, S. Le Coz, Analytical and Numerical Aspects of Partial Differential Equations, de Gruyter, Berlin, (2009), 151-192.
4) Crighton, D. G., Applications of KdV, Acta Applicandae Mathematicae 39: 39-67, 1995.